<a href="https://colab.research.google.com/github/carlosss1111/Neutrosophic_Decision_Making_Models/blob/main/Neutrosophic_Decision_Making_Models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
In this code we reproduces all manuscript tables (decision matrix, baselines, Method 1, Method 2, comparative summary)
and the robustness analyses (Dirichlet perturbations, LOO, grid sweeps, rank agreement, Kendall's W).
"""

from dataclasses import dataclass
from typing import Dict, Callable, Tuple, List, Iterable, Optional, Any
import math
import random
from collections import Counter, defaultdict

# ============================================================
# Global config
# ============================================================

SEED = 20260220

# Monte Carlo stability settings
N_LAMBDA_SAMPLES = 8000    # lambda perturbations
N_W_SAMPLES      = 8000    # criterion-weight perturbations
KAPPA_LAMBDA     = 200.0   # Dirichlet concentration for lambda (higher = smaller perturbations)
KAPPA_W          = 200.0   # Dirichlet concentration for w

# Grid sweeps (3-simplex).
GRID_N_LAMBDA = 20   # step = 1/20 = 0.05
GRID_N_W      = 20

# Ranking/ties tolerance
TOL = 1e-12

# Printing
PRINT_LATEX_TABLES = True
PRINT_STABILITY_LATEX_TABLES = True

# ============================================================
# Data structures
# ============================================================

@dataclass(frozen=True)
class SVNHFE:
    T: Tuple[float, ...]
    I: Tuple[float, ...]
    F: Tuple[float, ...]


# ============================================================
# Basic set helpers (F^*([0,1]) represented as sorted tuples)
# ============================================================

def hs(*vals: float) -> Tuple[float, ...]:
    return tuple(sorted({float(v) for v in vals}))

def set_union(A: Tuple[float, ...], B: Tuple[float, ...]) -> Tuple[float, ...]:
    return tuple(sorted(set(A).union(B)))

def set_inter(A: Tuple[float, ...], B: Tuple[float, ...]) -> Tuple[float, ...]:
    Bs = set(B)
    return tuple(sorted([a for a in A if a in Bs]))

def set_diff(A: Tuple[float, ...], B: Tuple[float, ...]) -> Tuple[float, ...]:
    Bs = set(B)
    return tuple(sorted([a for a in A if a not in Bs]))

def less_all(X: Tuple[float, ...], Y: Tuple[float, ...]) -> bool:

    if len(X) == 0 or len(Y) == 0:
        return True
    return max(X) < min(Y)

def leq0(A: Tuple[float, ...], B: Tuple[float, ...]) -> bool:

    return less_all(A, set_diff(B, A)) and less_all(set_diff(A, B), B)

# ============================================================
# Symmetric lattice operations on THFEs
# ============================================================

def meet0_thfe(A: Tuple[float, ...], B: Tuple[float, ...]) -> Tuple[float, ...]:

    U = set_union(A, B)
    inter = set_inter(A, B)
    m = max(min(A), min(B))
    symdiff = tuple(x for x in U if x not in set(inter))  # U \ (A ∩ B)
    Tset = tuple(x for x in symdiff if x >= m)
    if len(Tset) == 0:
        return U
    k = min(Tset)
    return tuple(x for x in U if x < k)

def join0_thfe(A: Tuple[float, ...], B: Tuple[float, ...]) -> Tuple[float, ...]:

    U = set_union(A, B)
    inter = set_inter(A, B)
    M = min(max(A), max(B))
    symdiff = tuple(x for x in U if x not in set(inter))  # U \ (A ∩ B)
    Sset = tuple(x for x in symdiff if x <= M)
    if len(Sset) == 0:
        return U
    h = max(Sset)
    return tuple(x for x in U if x > h)

# ============================================================
# Neutrosophic meet and join on SVNHFEs
# ============================================================

def meetN(A: SVNHFE, B: SVNHFE) -> SVNHFE:
    return SVNHFE(
        T=meet0_thfe(A.T, B.T),
        I=join0_thfe(A.I, B.I),
        F=join0_thfe(A.F, B.F),
    )

def joinN(A: SVNHFE, B: SVNHFE) -> SVNHFE:
    return SVNHFE(
        T=join0_thfe(A.T, B.T),
        I=meet0_thfe(A.I, B.I),
        F=meet0_thfe(A.F, B.F),
    )

def fold_meet(vals: List[SVNHFE]) -> SVNHFE:
    acc = vals[0]
    for v in vals[1:]:
        acc = meetN(acc, v)
    return acc

def fold_join(vals: List[SVNHFE]) -> SVNHFE:
    acc = vals[0]
    for v in vals[1:]:
        acc = joinN(acc, v)
    return acc

# ============================================================
# Base dominance functions on THFEs
# ============================================================

def ddf(A: Tuple[float, ...], B: Tuple[float, ...]) -> float:
    """
    Discrete dominance function (DDF):
      s(u,v)=1 if u<v, 1/2 if u=v, 0 otherwise
      S(A,B)=avg_{a in A,b in B} s(a,b)
    """
    total = 0.0
    for a in A:
        for b in B:
            if a < b:
                total += 1.0
            elif a == b:
                total += 0.5
    return total / (len(A) * len(B))

def score_mean(A: Tuple[float, ...]) -> float:
    return sum(A) / len(A)

def score_geomean(A: Tuple[float, ...]) -> float:

    if any(x <= 0.0 for x in A):
        return 0.0
    return math.exp(sum(math.log(x) for x in A) / len(A))

def score_max(A: Tuple[float, ...]) -> float:
    return max(A)

def score_min(A: Tuple[float, ...]) -> float:
    return min(A)

def df_from_score(score: Callable[[Tuple[float, ...]], float]) -> Callable[[Tuple[float, ...], Tuple[float, ...]], float]:
    """Score-induced DF: D_sigma(A,B)=1/2(1 + sigma(B) - sigma(A))."""
    def D(A: Tuple[float, ...], B: Tuple[float, ...]) -> float:
        val = 0.5 * (1.0 + score(B) - score(A))
        return max(0.0, min(1.0, val))
    return D

rdf   = df_from_score(score_mean)      # NRDF (RDF(mean))
gmdf  = df_from_score(score_geomean)   # DF(geomean)
maxdf = df_from_score(score_max)       # DF(max)
mindf = df_from_score(score_min)       # DF(min)

# ============================================================
# Neutrosophic lift of a DF on THFEs
# ============================================================

def neutro_df(A: SVNHFE, B: SVNHFE,
              S: Callable[[Tuple[float, ...], Tuple[float, ...]], float],
              lam: Tuple[float, float, float]) -> float:

    lamT, lamI, lamF = lam
    termT = S(A.T, B.T)
    termI = 1.0 - S(A.I, B.I)
    termF = 1.0 - S(A.F, B.F)
    return lamT * termT + lamI * termI + lamF * termF

# ============================================================
# Benchmark dataset
# ============================================================

def build_benchmark() -> Tuple[List[str], List[str], Dict[str, Dict[str, SVNHFE]]]:
    ALT = ["A1", "A2", "A3", "A4"]
    CRI = ["C1", "C2", "C3"]
    X: Dict[str, Dict[str, SVNHFE]] = {
        "A1": {"C1": SVNHFE(hs(0.3,0.4,0.5), hs(0.1),      hs(0.3,0.4)),
               "C2": SVNHFE(hs(0.5,0.6),     hs(0.2,0.3),  hs(0.3,0.4)),
               "C3": SVNHFE(hs(0.2,0.3),     hs(0.1,0.2),  hs(0.5,0.6))},
        "A2": {"C1": SVNHFE(hs(0.6,0.7),     hs(0.1,0.2),  hs(0.2,0.3)),
               "C2": SVNHFE(hs(0.6,0.7),     hs(0.1),      hs(0.3)),
               "C3": SVNHFE(hs(0.6,0.7),     hs(0.1,0.2),  hs(0.1,0.2))},
        "A3": {"C1": SVNHFE(hs(0.5,0.6),     hs(0.4),      hs(0.2,0.3)),
               "C2": SVNHFE(hs(0.6),         hs(0.3),      hs(0.4)),
               "C3": SVNHFE(hs(0.5,0.6),     hs(0.1),      hs(0.3))},
        "A4": {"C1": SVNHFE(hs(0.7,0.8),     hs(0.1),      hs(0.1,0.2)),
               "C2": SVNHFE(hs(0.6,0.7),     hs(0.1),      hs(0.2)),
               "C3": SVNHFE(hs(0.3,0.5),     hs(0.2),      hs(0.1,0.2,0.3))}
    }
    return ALT, CRI, X

# ============================================================
# Methods 1 and 2
# ============================================================

def compute_baselines(Xmat: Dict[str, Dict[str, SVNHFE]], ALT: List[str], CRI: List[str]) -> Tuple[Dict[str, SVNHFE], Dict[str, SVNHFE]]:
    Y = {c: fold_meet([Xmat[a][c] for a in ALT]) for c in CRI}
    Z = {c: fold_join([Xmat[a][c] for a in ALT]) for c in CRI}
    return Y, Z

def method1(Xmat: Dict[str, Dict[str, SVNHFE]],
            ALT: List[str], CRI: List[str],
            Y: Dict[str, SVNHFE], Z: Dict[str, SVNHFE],
            w: Dict[str, float], lam: Tuple[float,float,float],
            base_df: Callable[[Tuple[float,...],Tuple[float,...]], float]) -> Dict[str, Dict[str, Any]]:

    d_wedge = {a:{} for a in ALT}
    d_vee   = {a:{} for a in ALT}
    score_wedge = {}
    clos_vee = {}

    for a in ALT:
        sw = 0.0
        sv = 0.0
        for c in CRI:
            dw = neutro_df(Y[c], Xmat[a][c], base_df, lam)
            dv = neutro_df(Xmat[a][c], Z[c], base_df, lam)
            d_wedge[a][c] = dw
            d_vee[a][c] = dv
            sw += w[c] * dw
            sv += w[c] * dv
        score_wedge[a] = sw
        clos_vee[a] = 1.0 - sv

    return {
        "d_wedge": d_wedge,
        "score_wedge": score_wedge,
        "d_vee": d_vee,
        "clos_vee": clos_vee
    }

def method2_pref_agg(Xmat: Dict[str, Dict[str, SVNHFE]],
                     ALT: List[str], CRI: List[str],
                     w: Dict[str, float], lam: Tuple[float,float,float],
                     base_df: Callable[[Tuple[float,...],Tuple[float,...]], float]) -> Tuple[Dict[str, List[List[float]]], Dict[str, float]]:

    m = len(ALT)
    P = {c: [[0.0]*m for _ in range(m)] for c in CRI}
    for c in CRI:
        for i, ai in enumerate(ALT):
            for k, ak in enumerate(ALT):
                P[c][i][k] = neutro_df(Xmat[ak][c], Xmat[ai][c], base_df, lam)

    R = [[0.0]*m for _ in range(m)]
    for c in CRI:
        for i in range(m):
            for k in range(m):
                R[i][k] += w[c] * P[c][i][k]

    score_pref = {}
    for i, ai in enumerate(ALT):
        score_pref[ai] = sum(R[i][k] for k in range(m) if k != i) / (m - 1)

    return P, score_pref

# ============================================================
#   Ye baselines: SVNHFWA / SVNHFWG + cosine to ideal
# ============================================================

def _dedup_sorted(vals: Iterable[float], nd_key: int = 12) -> Tuple[float, ...]:

    seen = {}
    for x in vals:
        k = round(float(x), nd_key)
        if k not in seen:
            seen[k] = float(x)
    return tuple(sorted(seen.values()))

def ye_svnhfwa(els: List[SVNHFE], weights: List[float]) -> SVNHFE:

    if len(els) != len(weights):
        raise ValueError("SVNHFWA: len(els) must match len(weights).")
    k = len(els)

    T_out = []
    for gamma0 in els[0].T:
        for gamma1 in els[1].T:
            for gamma2 in els[2].T:
                gammas = [gamma0, gamma1, gamma2]
                prod = 1.0
                for j in range(k):
                    prod *= (1.0 - gammas[j]) ** weights[j]
                T_out.append(1.0 - prod)

    I_out = []
    for d0 in els[0].I:
        for d1 in els[1].I:
            for d2 in els[2].I:
                ds = [d0, d1, d2]
                prod = 1.0
                for j in range(k):
                    prod *= (ds[j] ** weights[j])
                I_out.append(prod)

    F_out = []
    for e0 in els[0].F:
        for e1 in els[1].F:
            for e2 in els[2].F:
                es = [e0, e1, e2]
                prod = 1.0
                for j in range(k):
                    prod *= (es[j] ** weights[j])
                F_out.append(prod)

    return SVNHFE(T=_dedup_sorted(T_out), I=_dedup_sorted(I_out), F=_dedup_sorted(F_out))

def ye_svnhfwg(els: List[SVNHFE], weights: List[float]) -> SVNHFE:

    if len(els) != len(weights):
        raise ValueError("SVNHFWG: len(els) must match len(weights).")
    k = len(els)

    T_out = []
    for gamma0 in els[0].T:
        for gamma1 in els[1].T:
            for gamma2 in els[2].T:
                gammas = [gamma0, gamma1, gamma2]
                prod = 1.0
                for j in range(k):
                    prod *= (gammas[j] ** weights[j])
                T_out.append(prod)

    I_out = []
    for d0 in els[0].I:
        for d1 in els[1].I:
            for d2 in els[2].I:
                ds = [d0, d1, d2]
                prod = 1.0
                for j in range(k):
                    prod *= (1.0 - ds[j]) ** weights[j]
                I_out.append(1.0 - prod)

    F_out = []
    for e0 in els[0].F:
        for e1 in els[1].F:
            for e2 in els[2].F:
                es = [e0, e1, e2]
                prod = 1.0
                for j in range(k):
                    prod *= (1.0 - es[j]) ** weights[j]
                F_out.append(1.0 - prod)

    return SVNHFE(T=_dedup_sorted(T_out), I=_dedup_sorted(I_out), F=_dedup_sorted(F_out))

def mean_tuple(A: Tuple[float, ...]) -> float:
    return sum(A) / len(A)

def ye_cosine_to_ideal(n: SVNHFE) -> float:

    aT, aI, aF = mean_tuple(n.T), mean_tuple(n.I), mean_tuple(n.F)
    den = math.sqrt(aT*aT + aI*aI + aF*aF)
    if den <= 0.0:
        return 0.0
    return aT / den

# ============================================================
# Ranking + ties-aware rank vectors
# ============================================================

def grouped_ranking_from_scores(scores: Dict[str, float], descending: bool = True, tol: float = TOL) -> List[List[str]]:

    items = sorted(scores.items(), key=lambda kv: (-kv[1], kv[0]) if descending else (kv[1], kv[0]))
    groups: List[List[str]] = []
    last_v: Optional[float] = None
    for a, v in items:
        if last_v is None or abs(v - last_v) > tol:
            groups.append([a])
            last_v = v
        else:
            groups[-1].append(a)
    return groups

def ranking_string(groups: List[List[str]]) -> str:
    return ">".join(["=".join(g) for g in groups])

def midranks_from_groups(groups: List[List[str]]) -> Dict[str, float]:
    out: Dict[str, float] = {}
    r = 1
    for g in groups:
        k = len(g)
        avg = (r + (r + k - 1)) / 2.0
        for a in g:
            out[a] = avg
        r += k
    return out

def footrule_distance(r1: Dict[str, float], r2: Dict[str, float]) -> float:
    return sum(abs(r1[a] - r2[a]) for a in r1)

# ============================================================
# Kendall
# ============================================================

def _cmp(x: float, y: float, tol: float = TOL) -> int:
    if abs(x - y) <= tol:
        return 0
    return 1 if x > y else -1

def kendall_tau_b_from_scores(scores1: Dict[str, float], scores2: Dict[str, float], items: List[str], tol: float = TOL) -> Tuple[float, int, int, int, int]:

    C = D = T1 = T2 = 0
    n = len(items)
    for i in range(n):
        for j in range(i+1, n):
            a, b = items[i], items[j]
            s1 = _cmp(scores1[a], scores1[b], tol)
            s2 = _cmp(scores2[a], scores2[b], tol)
            if s1 == 0 and s2 == 0:
                T1 += 1
                T2 += 1
            elif s1 == 0:
                T1 += 1
            elif s2 == 0:
                T2 += 1
            else:
                if s1 == s2:
                    C += 1
                else:
                    D += 1
    denom = math.sqrt((C + D + T1) * (C + D + T2))
    tau = 0.0 if denom == 0.0 else (C - D) / denom
    return tau, C, D, T1, T2

def kendall_distance_from_scores(scores1: Dict[str, float], scores2: Dict[str, float], items: List[str], tol: float = TOL) -> float:
    D = 0
    effective = 0
    n = len(items)
    for i in range(n):
        for j in range(i+1, n):
            a, b = items[i], items[j]
            s1 = _cmp(scores1[a], scores1[b], tol)
            s2 = _cmp(scores2[a], scores2[b], tol)
            if s1 == 0 and s2 == 0:
                continue
            effective += 1
            if s1 != 0 and s2 != 0 and s1 != s2:
                D += 1
    return 0.0 if effective == 0 else D / float(effective)

# ============================================================
# Spearman
# ============================================================

def spearman_rho_from_scores(scores1: Dict[str, float], scores2: Dict[str, float], items: List[str], tol: float = TOL) -> float:
    g1 = grouped_ranking_from_scores(scores1, descending=True, tol=tol)
    g2 = grouped_ranking_from_scores(scores2, descending=True, tol=tol)
    r1 = midranks_from_groups(g1)
    r2 = midranks_from_groups(g2)
    xs = [r1[a] for a in items]
    ys = [r2[a] for a in items]
    mx = sum(xs) / len(xs)
    my = sum(ys) / len(ys)
    num = sum((x - mx) * (y - my) for x, y in zip(xs, ys))
    denx = math.sqrt(sum((x - mx) ** 2 for x in xs))
    deny = math.sqrt(sum((y - my) ** 2 for y in ys))
    if denx == 0.0 or deny == 0.0:
        return 0.0
    return num / (denx * deny)

# ============================================================
# Kendall's W with tie correction
# ============================================================

def kendalls_W(rankings: List[Dict[str, float]], items: List[str]) -> float:
    """
    Tie-corrected Kendall's W:
      W = 12*S / (m^2*(n^3-n) - m*T)
    """
    m = len(rankings)
    n = len(items)
    sum_r = {a: 0.0 for a in items}
    for rv in rankings:
        for a in items:
            sum_r[a] += rv[a]
    Rbar = m * (n + 1) / 2.0
    S = sum((sum_r[a] - Rbar) ** 2 for a in items)

    Tcorr = 0.0
    for rv in rankings:
        buckets = defaultdict(list)
        for a in items:
            buckets[rv[a]].append(a)
        for group in buckets.values():
            t = len(group)
            if t > 1:
                Tcorr += (t**3 - t)

    denom = (m * m * (n**3 - n) - m * Tcorr)
    if denom <= 0.0:
        return 0.0
    return 12.0 * S / denom

# ============================================================
# Dirichlet + simplex grids
# ============================================================

def dirichlet_sample(rng: random.Random, alpha: List[float]) -> List[float]:
    xs = [rng.gammavariate(a, 1.0) for a in alpha]
    s = sum(xs)
    return [x / s for x in xs]

def simplex_grid_3(N: int) -> List[Tuple[float, float, float]]:
    """Barycentric grid on 3-simplex: (i,j,k)/N with i+j+k=N."""
    out = []
    for i in range(N + 1):
        for j in range(N + 1 - i):
            k = N - i - j
            out.append((i / N, j / N, k / N))
    return out

# ============================================================
# LaTeX
# ============================================================

def fmt_num(x: float, nd: int) -> str:
    return f"{x:.{nd}f}"

def fmt_set_tex(A: Tuple[float, ...], nd: int = 1) -> str:
    inside = ",".join(fmt_num(a, nd).rstrip("0").rstrip(".") if nd > 0 else str(int(a)) for a in A)
    return r"\{" + inside + r"\}"

def fmt_svnhfe_tex(X: SVNHFE, nd: int = 1) -> str:
    return rf"$\langle {fmt_set_tex(X.T, nd)},{fmt_set_tex(X.I, nd)},{fmt_set_tex(X.F, nd)}\rangle$"

def latex_table_svnhfs_matrix(Xmat: Dict[str, Dict[str, SVNHFE]], ALT: List[str], CRI: List[str]) -> str:
    lines = []
    lines.append(r"\begin{table}[htbp]")
    lines.append(r"\centering")
    lines.append(r"\caption{Single-valued neutrosophic hesitant fuzzy decision matrix.}")
    lines.append(r"\label{tab:svnhfs-matrix}")
    lines.append(r"\renewcommand{\arraystretch}{1.15}")
    lines.append(r"\setlength{\tabcolsep}{6pt}")
    lines.append(r"\begin{tabular}{c|c|c|c}")
    lines.append(r" & $C_{1}$ & $C_{2}$ & $C_{3}$ \\ \hline")
    for a in ALT:
        row = [rf"${a}$"]
        for c in CRI:
            row.append(fmt_svnhfe_tex(Xmat[a][c], nd=1))
        lines.append(" & ".join(row) + r"\\")
    lines.append(r"\end{tabular}")
    lines.append(r"\end{table}")
    return "\n".join(lines)

def latex_table_baselines(Y: Dict[str, SVNHFE], Z: Dict[str, SVNHFE], CRI: List[str]) -> str:
    lines = []
    lines.append(r"\begin{table}[htbp]")
    lines.append(r"\centering")
    lines.append(r"\caption{Criterion-wise baselines for Method~1: $\mathbf{Y}_j=\bigwedge_i X_{ij}$ and $\mathbf{Z}_j=\bigvee_i X_{ij}$.}")
    lines.append(r"\label{tab:svnhfs-baselines}")
    lines.append(r"\renewcommand{\arraystretch}{1.15}")
    lines.append(r"\setlength{\tabcolsep}{6pt}")
    lines.append(r"\begin{tabular}{c|c|c}")
    lines.append(r" & $\mathbf{Y}_j$ (infimum) & $\mathbf{Z}_j$ (supremum)\\ \hline")
    for c in CRI:
        lines.append(rf"${c}$ & {fmt_svnhfe_tex(Y[c], nd=1)} & {fmt_svnhfe_tex(Z[c], nd=1)}\\")
    lines.append(r"\end{tabular}")
    lines.append(r"\end{table}")
    return "\n".join(lines)

def latex_nested_method1_variant(d_by_alt: Dict[str, Dict[str, float]],
                                 total_by_alt: Dict[str, float],
                                 CRI: List[str],
                                 total_label_tex: str,
                                 nd: int = 6) -> str:
    lines = []
    lines.append(r"\begin{tabular}{c|ccc|c}")
    lines.append(r"\toprule")
    lines.append(rf" & $C_1$ & $C_2$ & $C_3$ & ${total_label_tex}$\\")
    lines.append(r"\midrule")
    for a in ["A1", "A2", "A3", "A4"]:
        vals = [fmt_num(d_by_alt[a][c], nd) for c in CRI]
        tot = fmt_num(total_by_alt[a], nd)
        lines.append(rf"${a}$ & " + " & ".join(vals) + rf" & {tot}\\")
    lines.append(r"\bottomrule")
    lines.append(r"\end{tabular}")
    return "\n".join(lines)

def latex_table_method1(all_results_m1: Dict[str, Dict[str, Any]], CRI: List[str]) -> str:
    order = ["NDDF", "NRDF", "DF(geomean)", "DF(max)", "DF(min)"]
    lines = []
    lines.append(r"\begin{table*}[!t]")
    lines.append(r"\centering")
    lines.append(r"\caption{Method~1 results for NDDF, NRDF (RDF(mean)), and score-induced DFs (DF(geomean), DF(max), DF(min)). Left: infimum-baseline $\mathrm{Score}^{\wedge}(A_i)$; right: supremum-baseline $\mathrm{Closeness}^{\vee}(A_i)$.}")
    lines.append(r"\label{tab:svnhfs-m1-results}")
    lines.append(r"\scriptsize")
    lines.append(r"\renewcommand{\arraystretch}{1.05}")
    lines.append(r"\setlength{\tabcolsep}{4.3pt}")
    lines.append(r"\begin{tabular}{@{}l@{\hspace{6pt}}c@{\hspace{10pt}}c@{}}")
    lines.append(r"\toprule")
    lines.append(r" & \textbf{Infimum-baseline $\mathrm{Score}^{\wedge}$} & \textbf{Supremum-baseline $\mathrm{Closeness}^{\vee}$} \\")
    lines.append(r"\midrule")
    for v in order:
        r = all_results_m1[v]
        left = latex_nested_method1_variant(
            r["d_wedge"], r["score_wedge"], CRI,
            total_label_tex=r"\mathrm{Score}^{\wedge}(A_i)", nd=6
        )
        right = latex_nested_method1_variant(
            r["d_vee"], r["clos_vee"], CRI,
            total_label_tex=r"\mathrm{Closeness}^{\vee}(A_i)", nd=6
        )
        lines.append(rf"\textbf{{{v}}} &")
        lines.append(left)
        lines.append(r"&")
        lines.append(right)
        lines.append(r"\\[6pt]")
    lines.append(r"\bottomrule")
    lines.append(r"\end{tabular}")
    lines.append(r"\end{table*}")
    return "\n".join(lines)

def latex_nested_pref_matrix(P: List[List[float]], nd: int = 4) -> str:
    lines = []
    lines.append(r"\begin{tabular}{c|cccc}")
    lines.append(r" & $A_1$ & $A_2$ & $A_3$ & $A_4$\\ \hline")
    for i, ai in enumerate(["A1", "A2", "A3", "A4"]):
        row = " & ".join(fmt_num(P[i][k], nd) for k in range(4))
        lines.append(rf"${ai}$ & {row}\\")
    lines.append(r"\end{tabular}")
    return "\n".join(lines)

def latex_table_method2_panel(all_P: Dict[str, Dict[str, List[List[float]]]], CRI: List[str]) -> str:
    order = ["NDDF", "NRDF", "DF(geomean)", "DF(max)", "DF(min)"]
    lines = []
    lines.append(r"\begin{table*}[!t]")
    lines.append(r"\centering")
    lines.append(r"\caption{Method~2: criterion-wise preference matrices $P^{(j)}=(p^{(j)}_{ik})$ for NDDF, NRDF (RDF(mean)), and score-induced DFs, with $p^{(j)}_{ik}=D^{\mathsf{N}}(X_{kj},X_{ij})$.}")
    lines.append(r"\label{tab:svnhfs-m2-nddf-nrdf-panels}")
    lines.append(r"\scriptsize")
    lines.append(r"\renewcommand{\arraystretch}{1.05}")
    lines.append(r"\setlength{\tabcolsep}{3.2pt}")
    lines.append(r"\resizebox{\textwidth}{!}{%")
    lines.append(r"\begin{tabular}{l|c|c|c}")
    lines.append(r"\toprule")
    lines.append(r"\textbf{Variant} & \textbf{$C_1$} & \textbf{$C_2$} & \textbf{$C_3$}\\")
    lines.append(r"\midrule")
    for v in order:
        lines.append(rf"\textbf{{{v}}} &")
        lines.append(latex_nested_pref_matrix(all_P[v]["C1"], nd=4))
        lines.append(r"&")
        lines.append(latex_nested_pref_matrix(all_P[v]["C2"], nd=4))
        lines.append(r"&")
        lines.append(latex_nested_pref_matrix(all_P[v]["C3"], nd=4))
        lines.append(r"\\")
        if v != order[-1]:
            lines.append(r"\midrule")
    lines.append(r"\bottomrule")
    lines.append(r"\end{tabular}%")
    lines.append(r"}")
    lines.append(r"\end{table*}")
    return "\n".join(lines)

def latex_table_method2_final(all_scores_pref: Dict[str, Dict[str, float]], ALT: List[str]) -> str:
    order = ["NDDF", "NRDF", "DF(geomean)", "DF(max)", "DF(min)"]
    lines = []
    lines.append(r"\begin{table*}[!t]")
    lines.append(r"\centering")
    lines.append(r"\caption{Method~2 (preference aggregation): final scores for NDDF/NRDF and for score-induced DFs.}")
    lines.append(r"\label{tab:svnhfs-m2-final-nddf-nrdf}")
    lines.append(r"\renewcommand{\arraystretch}{1.15}")
    lines.append(r"\setlength{\tabcolsep}{7pt}")
    lines.append(r"\begin{tabular}{l|cccc|l}")
    lines.append(r"\toprule")
    lines.append(r"\textbf{Variant} & $A_1$ & $A_2$ & $A_3$ & $A_4$ & \textbf{Ranking}\\")
    lines.append(r"\midrule")
    for v in order:
        sc = all_scores_pref[v]
        groups = grouped_ranking_from_scores(sc, descending=True, tol=TOL)
        rnk = ranking_string(groups)
        lines.append(
            rf"{v} & "
            + " & ".join(fmt_num(sc[a], 6) for a in ALT)
            + rf" & ${rnk}$\\"
        )
    lines.append(r"\bottomrule")
    lines.append(r"\end{tabular}")
    lines.append(r"\end{table*}")
    return "\n".join(lines)

def latex_table_comparative_summary(
    topsis_case1: Dict[str, float],
    all_results_m1: Dict[str, Dict[str, Any]],
    all_scores_pref: Dict[str, Dict[str, float]],
    ALT: List[str],
    w_tex: str,
    lam_tex: str
) -> str:
    lines = []
    lines.append(r"\begin{table*}[!t]")
    lines.append(r"\centering")
    lines.append(
        r"\caption{Comparative summary on the SVNHFS benchmark (four alternatives, three criteria). "
        r"TOPSIS baseline reports relative closeness $RC(A_i)$ (Case~1). "
        r"Our results use TOPSIS criterion weights $w^N="
        + w_tex +
        r"$ and neutrosophic component weights $\lambda="
        + lam_tex +
        r"$.}"
    )
    lines.append(r"\label{tab:svnhfs-comparative-summary}")
    lines.append(r"\resizebox{\textwidth}{!}{%")
    lines.append(r"\begin{tabular}{l l cccc l}")
    lines.append(r"\toprule")
    lines.append(r"Variant & Output & $A_1$ & $A_2$ & $A_3$ & $A_4$ & Ranking \\")
    lines.append(r"\midrule")

    groups_top = grouped_ranking_from_scores(topsis_case1, descending=True, tol=1e-9)
    rnk_top = ranking_string(groups_top)
    lines.append(
        r"TOPSIS (Case 1) & $RC(A_i)$ & "
        + " & ".join(fmt_num(topsis_case1[a], 3) for a in ALT)
        + rf" & ${rnk_top}$\\"
    )
    lines.append(r"\midrule")

    def add_block(v: str, label_variant: str):
        m1 = all_results_m1[v]
        sc1 = m1["score_wedge"]
        sc2 = m1["clos_vee"]
        sc3 = all_scores_pref[v]
        rnk1 = ranking_string(grouped_ranking_from_scores(sc1, True, TOL))
        rnk2 = ranking_string(grouped_ranking_from_scores(sc2, True, TOL))
        rnk3 = ranking_string(grouped_ranking_from_scores(sc3, True, TOL))
        lines.append(
            rf"{label_variant} & Method 1: $\mathrm{{Score}}^{{\wedge}}$ & "
            + " & ".join(fmt_num(sc1[a], 6) for a in ALT)
            + rf" & ${rnk1}$\\"
        )
        lines.append(
            rf"{label_variant} & Method 1: $\mathrm{{Closeness}}^{{\vee}}$ & "
            + " & ".join(fmt_num(sc2[a], 6) for a in ALT)
            + rf" & ${rnk2}$\\"
        )
        lines.append(
            rf"{label_variant} & Method 2: pref.\ agg.\ (TOPSIS-$w$) & "
            + " & ".join(fmt_num(sc3[a], 6) for a in ALT)
            + rf" & ${rnk3}$\\"
        )

    add_block("NDDF", "DDF")
    lines.append(r"\cmidrule(lr){1-8}")
    add_block("NRDF", "RDF(mean)")
    lines.append(r"\cmidrule(lr){1-8}")
    add_block("DF(geomean)", "DF(geomean)")
    lines.append(r"\cmidrule(lr){1-8}")
    add_block("DF(max)", "DF(max)")
    lines.append(r"\cmidrule(lr){1-8}")
    add_block("DF(min)", "DF(min)")

    lines.append(r"\bottomrule")
    lines.append(r"\end{tabular}%")
    lines.append(r"}")
    lines.append(r"\end{table*}")
    return "\n".join(lines)

def latex_table_stability_summary(rows: List[Dict[str, Any]], caption: str, label: str) -> str:
    lines = []
    lines.append(r"\begin{table*}[!t]")
    lines.append(r"\centering")
    lines.append(rf"\caption{{{caption}}}")
    lines.append(rf"\label{{{label}}}")
    lines.append(r"\renewcommand{\arraystretch}{1.10}")
    lines.append(r"\setlength{\tabcolsep}{5.5pt}")
    lines.append(r"\begin{tabular}{l l ccccc}")
    lines.append(r"\toprule")
    lines.append(r"Method & Base rank & $p(\text{base})$ & $p(R_T)$ & $p(R_A)$ & $\bar{\tau}_b$ & $\bar{\rho}$\\")
    lines.append(r"\midrule")
    for r in rows:
        lines.append(
            rf"{r['method']} & ${r['base_rank']}$ & {fmt_num(r['p_base'],4)} & {fmt_num(r['p_RT'],4)} & {fmt_num(r['p_RA'],4)} & {fmt_num(r['mean_tau'],4)} & {fmt_num(r['mean_rho'],4)}\\"
        )
    lines.append(r"\bottomrule")
    lines.append(r"\end{tabular}")
    lines.append(r"\end{table*}")
    return "\n".join(lines)

# ============================================================
#  Robustness and   stability
# ============================================================

def entropy_from_counts(cnt: Counter) -> float:
    tot = sum(cnt.values())
    if tot <= 0:
        return 0.0
    H = 0.0
    for v in cnt.values():
        p = v / tot
        H -= p * math.log(p + 1e-300)
    return H

def summarize_samples(method_name: str,
                      base_scores: Dict[str, float],
                      sampled_scores: List[Dict[str, float]],
                      items: List[str],
                      RT: str,
                      RA: str) -> Dict[str, Any]:
    base_groups = grouped_ranking_from_scores(base_scores, True, TOL)
    base_rank_str = ranking_string(base_groups)

    freq = Counter()
    taus = []
    rhos = []
    for sc in sampled_scores:
        rank_str = ranking_string(grouped_ranking_from_scores(sc, True, TOL))
        freq[rank_str] += 1
        tau, *_ = kendall_tau_b_from_scores(base_scores, sc, items, TOL)
        rho = spearman_rho_from_scores(base_scores, sc, items, TOL)
        taus.append(tau)
        rhos.append(rho)

    tot = len(sampled_scores)
    p_base = freq[base_rank_str] / tot if tot else 0.0
    p_RT = freq[RT] / tot if tot else 0.0
    p_RA = freq[RA] / tot if tot else 0.0

    return {
        "method": method_name,
        "base_rank": base_rank_str,
        "p_base": p_base,
        "p_RT": p_RT,
        "p_RA": p_RA,
        "entropy": entropy_from_counts(freq),
        "mean_tau": sum(taus) / tot if tot else 0.0,
        "mean_rho": sum(rhos) / tot if tot else 0.0,
        "freq": freq
    }

# ============================================================
#  Pairwise  comparison panels
# ============================================================

def print_pairwise_panel(methods: Dict[str, Dict[str, float]], items: List[str], title: str) -> None:
    names = list(methods.keys())
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)
    for i in range(len(names)):
        for j in range(i + 1, len(names)):
            a, b = names[i], names[j]
            tau, C, D, T1, T2 = kendall_tau_b_from_scores(methods[a], methods[b], items, TOL)
            kd = kendall_distance_from_scores(methods[a], methods[b], items, TOL)
            rho = spearman_rho_from_scores(methods[a], methods[b], items, TOL)
            ra = midranks_from_groups(grouped_ranking_from_scores(methods[a], True, TOL))
            rb = midranks_from_groups(grouped_ranking_from_scores(methods[b], True, TOL))
            fr = footrule_distance(ra, rb)
            print(f"- {a} vs {b}: tau_b={fmt_num(tau,6)}  dist={fmt_num(kd,6)}  rho={fmt_num(rho,6)}  footrule={fmt_num(fr,3)}  (C={C},D={D},T1={T1},T2={T2})")

# ============================================================
# Complexity estimates
# ========== ==================================================

def complexity_report(Xmat: Dict[str, Dict[str, SVNHFE]], ALT: List[str], CRI: List[str]) -> None:
    m = len(ALT)
    n = len(CRI)
    sizes = []
    for a in ALT:
        for c in CRI:
            x = Xmat[a][c]
            sizes.append((len(x.T), len(x.I), len(x.F)))
    avgT = sum(s[0] for s in sizes) / len(sizes)
    avgI = sum(s[1] for s in sizes) / len(sizes)
    avgF = sum(s[2] for s in sizes) / len(sizes)

    avg_sq = (avgT * avgT + avgI * avgI + avgF * avgF) / 3.0

    m1_pairs = m * n
    m2_pairs = m * m * n

    print("\n" + "=" * 80)
    print("COMPLEXITY (rough, operation-count scale)")
    print("=" * 80)
    print(f"m={m} alternatives, n={n} criteria.")
    print(f"Avg component sizes: |T|~{avgT:.2f}, |I|~{avgI:.2f}, |F|~{avgF:.2f}.")
    print(f"Approx DF kernel cost per component ~O(|A||B|); avg |A||B| scale ~{avg_sq:.2f}.")
    print(f"Method 1: DF evaluations ~ m*n = {m1_pairs} (each uses 3 components) => ~ {3*m1_pairs*avg_sq:.2f} kernel ops (scale).")
    print(f"Method 2: DF evaluations ~ m^2*n = {m2_pairs} (each uses 3 components) => ~ {3*m2_pairs*avg_sq:.2f} kernel ops (scale).")
    print("Baselines Y_j/Z_j computed once per criterion via closed-form meet/join on THFEs (linear in set sizes).")

# ============================================================
# Main
# ============================================================

def main() -> None:
    rng = random.Random(SEED)

    ALT, CRI, X = build_benchmark()


    w_base = {"C1": 0.35, "C2": 0.25, "C3": 0.40}
    lam_base = (1.0/3.0, 1.0/3.0, 1.0/3.0)

    # For LaTeX captions
    w_tex = r"(0.35,0.25,0.40)"
    lam_tex = r"(\tfrac13,\tfrac13,\tfrac13)"

    # TOPSIS baseline values (as in your current script/table)
    topsis_case1 = {"A1": 0.461, "A2": 0.826, "A3": 0.462, "A4": 0.796}

    # Dominant patterns
    RT = "A2>A4>A3>A1"
    RA = "A4>A2>A3>A1"

    # Variants
    VARIANTS: Dict[str, Callable[[Tuple[float,...],Tuple[float,...]], float]] = {
        "NDDF": ddf,
        "NRDF": rdf,
        "DF(geomean)": gmdf,
        "DF(max)": maxdf,
        "DF(min)": mindf,
    }

    # Baselines Y,Z from lattice
    Y, Z = compute_baselines(X, ALT, CRI)

    # Complexity
    complexity_report(X, ALT, CRI)

    # Run Methods 1 and 2 (with NEW lambda)
    all_results_m1: Dict[str, Dict[str, Any]] = {}
    all_P: Dict[str, Dict[str, List[List[float]]]] = {}
    all_scores_pref: Dict[str, Dict[str, float]] = {}

    for vname, base_df in VARIANTS.items():
        all_results_m1[vname] = method1(X, ALT, CRI, Y, Z, w_base, lam_base, base_df)
        P, sc = method2_pref_agg(X, ALT, CRI, w_base, lam_base, base_df)
        all_P[vname] = P
        all_scores_pref[vname] = sc

    # Ye (2015) baselines depend only on criterion weights w (unchanged)
    w_list = [w_base["C1"], w_base["C2"], w_base["C3"]]
    ye_scores_A = {}
    ye_scores_G = {}
    for a in ALT:
        els = [X[a][c] for c in CRI]
        aggA = ye_svnhfwa(els, w_list)
        aggG = ye_svnhfwg(els, w_list)
        ye_scores_A[a] = ye_cosine_to_ideal(aggA)
        ye_scores_G[a] = ye_cosine_to_ideal(aggG)

    ye_groups_A = grouped_ranking_from_scores(ye_scores_A, True, TOL)
    ye_groups_G = grouped_ranking_from_scores(ye_scores_G, True, TOL)

    # ============================================================
    # LaTeX tables
    # ============================================================

    if PRINT_LATEX_TABLES:
        print("\n% ============================================================")
        print("% LaTeX: Decision matrix (Table~svnhfs-matrix)")
        print("% ============================================================\n")
        print(latex_table_svnhfs_matrix(X, ALT, CRI))

        print("\n% ============================================================")
        print("% LaTeX: Baselines (Table~svnhfs-baselines)")
        print("% ============================================================\n")
        print(latex_table_baselines(Y, Z, CRI))

        print("\n% ============================================================")
        print("% LaTeX: Method 1 results (Table~svnhfs-m1-results)")
        print("% ============================================================\n")
        print(latex_table_method1(all_results_m1, CRI))

        print("\n% ============================================================")
        print("% LaTeX: Method 2 preference matrices panel (Table~svnhfs-m2-nddf-nrdf-panels)")
        print("% ============================================================\n")
        print(latex_table_method2_panel(all_P, CRI))

        print("\n% ============================================================")
        print("% LaTeX: Method 2 final scores (Table~svnhfs-m2-final-nddf-nrdf)")
        print("% ============================================================\n")
        print(latex_table_method2_final(all_scores_pref, ALT))

        print("\n% ============================================================")
        print("% LaTeX: Comparative summary (Table~svnhfs-comparative-summary)")
        print("% ============================================================\n")
        print(latex_table_comparative_summary(
            topsis_case1=topsis_case1,
            all_results_m1=all_results_m1,
            all_scores_pref=all_scores_pref,
            ALT=ALT,
            w_tex=w_tex,
            lam_tex=lam_tex
        ))

    # ============================================================
    # Summary
    # ============================================================

    print("\n" + "="*80)
    print("BASELINE RESULTS (paper w, OPTION B lambda)")
    print("="*80)
    print(f"Using w = {w_base}  and  lambda = {lam_base}")
    print(f"TOPSIS Case 1 ranking: {ranking_string(grouped_ranking_from_scores(topsis_case1, True, 1e-9))}  (RC={topsis_case1})")
    print(f"Ye SVNHFWA+cos ranking: {ranking_string(ye_groups_A)}  (cos={ {a: round(ye_scores_A[a],4) for a in ALT} })")
    print(f"Ye SVNHFWG+cos ranking: {ranking_string(ye_groups_G)}  (cos={ {a: round(ye_scores_G[a],4) for a in ALT} })")

    for v in ["NDDF", "NRDF", "DF(geomean)", "DF(max)", "DF(min)"]:
        m1 = all_results_m1[v]
        r1 = ranking_string(grouped_ranking_from_scores(m1["score_wedge"], True, TOL))
        r2 = ranking_string(grouped_ranking_from_scores(m1["clos_vee"], True, TOL))
        r3 = ranking_string(grouped_ranking_from_scores(all_scores_pref[v], True, TOL))
        print(f"{v:10s}  M1 Score^wedge: {r1:15s}  |  M1 Closeness^vee: {r2:15s}  |  M2 pref: {r3:15s}")

    # ============================================================
    # Pairwise rank agreement panel
    # ============================================================

    def mk_method_scores_dict() -> Dict[str, Dict[str, float]]:
        out = {}
        out["TOPSIS_C1"] = topsis_case1
        out["Ye_SVNHFWA_cos"] = ye_scores_A
        out["Ye_SVNHFWG_cos"] = ye_scores_G
        for v in ["NDDF", "NRDF", "DF(geomean)", "DF(max)", "DF(min)"]:
            out[f"{v}_M1_ScoreW"] = all_results_m1[v]["score_wedge"]
            out[f"{v}_M1_CloseV"] = all_results_m1[v]["clos_vee"]
            out[f"{v}_M2_Pref"] = all_scores_pref[v]
        return out

    methods_scores = mk_method_scores_dict()
    panel_names = [
        "TOPSIS_C1",
        "Ye_SVNHFWA_cos",
        "Ye_SVNHFWG_cos",
        "NRDF_M1_ScoreW",
        "NRDF_M2_Pref",
        "NDDF_M1_ScoreW",
        "NDDF_M2_Pref",
        "DF(max)_M1_ScoreW",
        "DF(max)_M2_Pref",
    ]
    reduced_panel = {k: methods_scores[k] for k in panel_names}
    print_pairwise_panel(reduced_panel, ALT, title="PAIRWISE RANK AGREEMENT (tau-b / rho / distances) — selected panel")

    W_rankvecs = []
    for k in panel_names:
        g = grouped_ranking_from_scores(methods_scores[k], True, TOL)
        W_rankvecs.append(midranks_from_groups(g))
    W = kendalls_W(W_rankvecs, ALT)
    print("\n" + "="*80)
    print(f"KENDALL'S W (ties-aware) across {len(panel_names)} rankings = {fmt_num(W,6)}")
    print("="*80)

    # ============================================================
    # Stability analysis A: lambda perturbations
    # ============================================================

    def sample_lambda(kappa: float) -> Tuple[float, float, float]:
        alpha = [
            max(1e-9, kappa * lam_base[0]),
            max(1e-9, kappa * lam_base[1]),
            max(1e-9, kappa * lam_base[2]),
        ]
        x = dirichlet_sample(rng, alpha)
        return (x[0], x[1], x[2])

    lambda_stability_rows = []
    method_specs_lambda: List[Tuple[str, Callable[[Tuple[float,float,float]], Dict[str,float]]]] = []

    for vname, base_df in VARIANTS.items():
        method_specs_lambda.append((
            f"{vname}-M1-ScoreW",
            lambda lam, df=base_df: method1(X, ALT, CRI, Y, Z, w_base, lam, df)["score_wedge"]
        ))
        method_specs_lambda.append((
            f"{vname}-M1-CloseV",
            lambda lam, df=base_df: method1(X, ALT, CRI, Y, Z, w_base, lam, df)["clos_vee"]
        ))
        method_specs_lambda.append((
            f"{vname}-M2-Pref",
            lambda lam, df=base_df: method2_pref_agg(X, ALT, CRI, w_base, lam, df)[1]
        ))

    print("\n" + "="*80)
    print(f"STABILITY A: lambda perturbation (Dirichlet, kappa={KAPPA_LAMBDA}, N={N_LAMBDA_SAMPLES})")
    print("="*80)

    for name, fn in method_specs_lambda:
        base_sc = fn(lam_base)
        samples = []
        for _ in range(N_LAMBDA_SAMPLES):
            lam_s = sample_lambda(KAPPA_LAMBDA)
            samples.append(fn(lam_s))
        summ = summarize_samples(name, base_sc, samples, ALT, RT, RA)
        lambda_stability_rows.append({
            "method": name,
            "base_rank": summ["base_rank"],
            "p_base": summ["p_base"],
            "p_RT": summ["p_RT"],
            "p_RA": summ["p_RA"],
            "entropy": summ["entropy"],
            "mean_tau": summ["mean_tau"],
            "mean_rho": summ["mean_rho"],
        })
        top3 = summ["freq"].most_common(3)
        print(f"- {name:18s} base={summ['base_rank']:15s}  p(base)={summ['p_base']:.4f}  "
              f"p(RT)={summ['p_RT']:.4f}  p(RA)={summ['p_RA']:.4f}  "
              f"tau={summ['mean_tau']:.4f} rho={summ['mean_rho']:.4f}  "
              f"top3={[(k, v/N_LAMBDA_SAMPLES) for k,v in top3]}")

    # ============================================================
    # Stability analysis B: criterion-weight perturbations
    # ============================================================

    def sample_w(kappa: float) -> Dict[str, float]:
        alpha = [
            max(1e-9, kappa * w_base["C1"]),
            max(1e-9, kappa * w_base["C2"]),
            max(1e-9, kappa * w_base["C3"]),
        ]
        x = dirichlet_sample(rng, alpha)
        return {"C1": x[0], "C2": x[1], "C3": x[2]}

    w_stability_rows = []
    method_specs_w: List[Tuple[str, Callable[[Dict[str,float]], Dict[str,float]]]] = []

    for vname, base_df in VARIANTS.items():
        method_specs_w.append((
            f"{vname}-M1-ScoreW",
            lambda w, df=base_df: method1(X, ALT, CRI, Y, Z, w, lam_base, df)["score_wedge"]
        ))
        method_specs_w.append((
            f"{vname}-M1-CloseV",
            lambda w, df=base_df: method1(X, ALT, CRI, Y, Z, w, lam_base, df)["clos_vee"]
        ))
        method_specs_w.append((
            f"{vname}-M2-Pref",
            lambda w, df=base_df: method2_pref_agg(X, ALT, CRI, w, lam_base, df)[1]
        ))

    def ye_scores_given_w(w: Dict[str, float], mode: str) -> Dict[str, float]:
        ww = [w["C1"], w["C2"], w["C3"]]
        out = {}
        for a in ALT:
            els = [X[a][c] for c in CRI]
            agg = ye_svnhfwa(els, ww) if mode == "SVNHFWA" else ye_svnhfwg(els, ww)
            out[a] = ye_cosine_to_ideal(agg)
        return out

    method_specs_w.append(("Ye-SVNHFWA-cos", lambda w: ye_scores_given_w(w, "SVNHFWA")))
    method_specs_w.append(("Ye-SVNHFWG-cos", lambda w: ye_scores_given_w(w, "SVNHFWG")))

    print("\n" + "="*80)
    print(f"STABILITY B: criterion-weight perturbation (Dirichlet, kappa={KAPPA_W}, N={N_W_SAMPLES})")
    print("="*80)

    for name, fn in method_specs_w:
        base_sc = fn(w_base)
        samples = []
        for _ in range(N_W_SAMPLES):
            w_s = sample_w(KAPPA_W)
            samples.append(fn(w_s))
        summ = summarize_samples(name, base_sc, samples, ALT, RT, RA)
        w_stability_rows.append({
            "method": name,
            "base_rank": summ["base_rank"],
            "p_base": summ["p_base"],
            "p_RT": summ["p_RT"],
            "p_RA": summ["p_RA"],
            "entropy": summ["entropy"],
            "mean_tau": summ["mean_tau"],
            "mean_rho": summ["mean_rho"],
        })
        top3 = summ["freq"].most_common(3)
        print(f"- {name:18s} base={summ['base_rank']:15s}  p(base)={summ['p_base']:.4f}  "
              f"p(RT)={summ['p_RT']:.4f}  p(RA)={summ['p_RA']:.4f}  "
              f"tau={summ['mean_tau']:.4f} rho={summ['mean_rho']:.4f}  "
              f"top3={[(k, v/N_W_SAMPLES) for k,v in top3]}")

    # ============================================================
    # LOO
    # ============================================================

    print("\n" + "="*80)
    print("LEAVE-ONE-CRITERION-OUT (LOO) SENSITIVITY")
    print("="*80)

    def renorm_w(crits_keep: List[str]) -> Dict[str, float]:
        s = sum(w_base[c] for c in crits_keep)
        return {c: (w_base[c] / s) for c in crits_keep}

    def restrict_X(crits_keep: List[str]) -> Tuple[List[str], Dict[str, Dict[str, SVNHFE]]]:
        Xk = {a: {c: X[a][c] for c in crits_keep} for a in ALT}
        return crits_keep, Xk

    for removed in CRI:
        keep = [c for c in CRI if c != removed]
        wk = renorm_w(keep)
        keepC, Xk = restrict_X(keep)
        Yk, Zk = compute_baselines(Xk, ALT, keepC)

        print(f"\n- Remove {removed}: keep={keep}, renorm w={ {c: round(wk[c],4) for c in keep} }")
        for vname, base_df in VARIANTS.items():
            r1 = method1(Xk, ALT, keepC, Yk, Zk, wk, lam_base, base_df)
            scW = r1["score_wedge"]
            scC = r1["clos_vee"]
            _, scP = method2_pref_agg(Xk, ALT, keepC, wk, lam_base, base_df)
            gW = ranking_string(grouped_ranking_from_scores(scW, True, TOL))
            gC = ranking_string(grouped_ranking_from_scores(scC, True, TOL))
            gP = ranking_string(grouped_ranking_from_scores(scP, True, TOL))
            print(f"  {vname:10s}  M1 ScoreW={gW:15s}  M1 CloseV={gC:15s}  M2 Pref={gP:15s}")

        yeA = ye_scores_given_w({**wk, removed: 0.0}, "SVNHFWA")
        yeG = ye_scores_given_w({**wk, removed: 0.0}, "SVNHFWG")
        print(f"  Ye SVNHFWA: {ranking_string(grouped_ranking_from_scores(yeA, True, TOL))}")
        print(f"  Ye SVNHFWG: {ranking_string(grouped_ranking_from_scores(yeG, True, TOL))}")

    # ============================================================
    # Grid sweeps on simplex
    # ============================================================

    print("\n" + "="*80)
    print("GRID SWEEP (simplex) — ranking frequencies")
    print("="*80)

    grid_lambda = simplex_grid_3(GRID_N_LAMBDA)
    grid_w = simplex_grid_3(GRID_N_W)


    rep_df = rdf

    def rep_scores_lambda(lam: Tuple[float, float, float]) -> Dict[str, float]:
        return method2_pref_agg(X, ALT, CRI, w_base, lam, rep_df)[1]

    freq_grid_lambda = Counter()
    for lam in grid_lambda:
        sc = rep_scores_lambda(lam)
        freq_grid_lambda[ranking_string(grouped_ranking_from_scores(sc, True, TOL))] += 1
    print(f"Lambda-grid ({len(grid_lambda)} points) for NRDF-M2-Pref:")
    for k, v in freq_grid_lambda.most_common(10):
        print(f"  {k:15s}  freq={v}  p={v/len(grid_lambda):.4f}")

    def rep_scores_w(wt: Dict[str, float]) -> Dict[str, float]:
        return method2_pref_agg(X, ALT, CRI, wt, lam_base, rep_df)[1]

    freq_grid_w = Counter()
    for (a, b, c) in grid_w:
        wt = {"C1": a, "C2": b, "C3": c}
        sc = rep_scores_w(wt)
        freq_grid_w[ranking_string(grouped_ranking_from_scores(sc, True, TOL))] += 1
    print(f"W-grid ({len(grid_w)} points) for NRDF-M2-Pref:")
    for k, v in freq_grid_w.most_common(10):
        print(f"  {k:15s}  freq={v}  p={v/len(grid_w):.4f}")

    # ============================================================
    # Optional LaTeX: Stability summary tables
    # ============================================================

    if PRINT_STABILITY_LATEX_TABLES:
        want = set()
        for v in ["NDDF", "NRDF", "DF(geomean)", "DF(max)", "DF(min)"]:
            want.add(f"{v}-M1-ScoreW")
            want.add(f"{v}-M2-Pref")

        lambda_rows_small = [r for r in lambda_stability_rows if r["method"] in want]
        w_rows_small = [r for r in w_stability_rows if (r["method"] in want or r["method"].startswith("Ye-"))]

        print("\n% ============================================================")
        print("% LaTeX: Stability summary (lambda perturbations)")
        print("% ============================================================\n")
        print(latex_table_stability_summary(
            lambda_rows_small,
            caption=(
                r"Stability under neutrosophic-component weight perturbations. "
                r"Dirichlet sampling around $\lambda=(\tfrac13,\tfrac13,\tfrac13)$ "
                rf"with concentration $\kappa={KAPPA_LAMBDA}$ and $N={N_LAMBDA_SAMPLES}$ draws. "
                r"We report the probability of reproducing the base ranking, the frequencies of the two dominant patterns "
                r"$R_T$ and $R_A$, and mean Kendall $\tau_b$ and Spearman $\rho$ with respect to the base ranking."
            ),
            label="tab:stability-lambda"
        ))

        print("\n% ============================================================")
        print("% LaTeX: Stability summary (criterion-weight perturbations)")
        print("% ============================================================\n")
        print(latex_table_stability_summary(
            w_rows_small,
            caption=(
                r"Stability under criterion-weight perturbations. "
                r"Dirichlet sampling around $w=(0.35,0.25,0.40)$ "
                rf"with concentration $\kappa={KAPPA_W}$ and $N={N_W_SAMPLES}$ draws. "
                r"We report the probability of reproducing the base ranking, the frequencies of the two dominant patterns "
                r"$R_T$ and $R_A$, and mean Kendall $\tau_b$ and Spearman $\rho$ with respect to the base ranking."
            ),
            label="tab:stability-w"
        ))

    print("\nDONE.")

if __name__ == "__main__":
    main()


COMPLEXITY (rough, operation-count scale)
m=4 alternatives, n=3 criteria.
Avg component sizes: |T|~2.00, |I|~1.33, |F|~1.75.
Approx DF kernel cost per component ~O(|A||B|); avg |A||B| scale ~2.95.
Method 1: DF evaluations ~ m*n = 12 (each uses 3 components) => ~ 106.08 kernel ops (scale).
Method 2: DF evaluations ~ m^2*n = 48 (each uses 3 components) => ~ 424.33 kernel ops (scale).
Baselines Y_j/Z_j computed once per criterion via closed-form meet/join on THFEs (linear in set sizes).

% ============================================================
% LaTeX: Decision matrix (Table~svnhfs-matrix)
% ============================================================

\begin{table}[htbp]
\centering
\caption{Single-valued neutrosophic hesitant fuzzy decision matrix.}
\label{tab:svnhfs-matrix}
\renewcommand{\arraystretch}{1.15}
\setlength{\tabcolsep}{6pt}
\begin{tabular}{c|c|c|c}
 & $C_{1}$ & $C_{2}$ & $C_{3}$ \\ \hline
$A1$ & $\langle \{0.3,0.4,0.5\},\{0.1\},\{0.3,0.4\}\rangle$ & $\langle \{0.5,0.6